In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense
from keras import Sequential

from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# One-time downloads for nltk's tokenizer, stopword list, and lemmatizer dictionary

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [5]:
df = pd.read_csv('/content/smartcart_customer_queries.csv')
print(df.shape)
df.head(10)

(420, 3)


,query_id,query_text,category
0,1,kids dress,Apparel
1,2,Organic Sugar Nestle,Grocery
2,3,KIDS DRESS GREY SIZE 8,Apparel
3,4,SUGAR FREE BISCUITS,Grocery
4,5,sugar free peanut butter,Grocery
5,6,headphones,Electronics
6,7,GAMING MOUSE BELOW 5000,Electronics
7,8,gaming mouse between 3000 and 6000...,Electronics
8,9,Wireless Earbuds Boat,Electronics
9,10,waterproof power bank JBL,Electronics


In [6]:
df['category'].value_counts()

,count
category,
Apparel,140
Grocery,140
Electronics,140


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['query_text'].apply(clean_text)
df[['query_text', 'clean_text']].sample(5)

,query_text,clean_text
145,delivery was late but the smartwatch quality i...,delivery was late but the smartwatch quality i...
334,men's party wear gown navy blue,men s party wear gown navy blue
175,wireless charger cable,wireless charger cable
369,latest smart bulb,latest smart bulb
416,best wifi router between 3000 and 6000,best wifi router between 3000 and 6000


In [9]:
df['tokens'] = df['clean_text'].apply(word_tokenize)
df[['clean_text', 'tokens']].sample(5, random_state=1)

,clean_text,tokens
378,packaging was damaged but the sneakers itself ...,"[packaging, was, damaged, but, the, sneakers, ..."
165,1kg pack organic honey aashirvaad,"[1kg, pack, organic, honey, aashirvaad]"
17,hoodie,[hoodie]
179,gluten free spices combo nestle,"[gluten, free, spices, combo, nestle]"
4,sugar free peanut butter,"[sugar, free, peanut, butter]"


In [10]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df['tokens_no_stop'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stop']].sample(5, random_state=1)

,tokens,tokens_no_stop
378,"[packaging, was, damaged, but, the, sneakers, ...","[packaging, damaged, sneakers, fine]"
165,"[1kg, pack, organic, honey, aashirvaad]","[1kg, pack, organic, honey, aashirvaad]"
17,[hoodie],[hoodie]
179,"[gluten, free, spices, combo, nestle]","[gluten, free, spices, combo, nestle]"
4,"[sugar, free, peanut, butter]","[sugar, free, peanut, butter]"


In [11]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

sample_words = ['running', 'shoes', 'organic', 'cheaper', 'earbuds', 'jackets']

comparison = pd.DataFrame({
    'word': sample_words,
    'stemmed': [stemmer.stem(w) for w in sample_words],
    'lemmatized': [lemmatizer.lemmatize(w) for w in sample_words]})
comparison

,word,stemmed,lemmatized
0,running,run,running
1,shoes,shoe,shoe
2,organic,organ,organic
3,cheaper,cheaper,cheaper
4,earbuds,earbud,earbuds
5,jackets,jacket,jacket


In [12]:
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

df['final_tokens'] = df['tokens_no_stop'].apply(lemmatize_tokens)
df['processed_text'] = df['final_tokens'].apply(lambda toks: ' '.join(toks))
df[['query_text', 'processed_text']].sample(8, random_state=1)

,query_text,processed_text
378,packaging was damaged but the sneakers itself ...,packaging damaged sneaker fine
165,1KG PACK ORGANIC HONEY AASHIRVAAD,1kg pack organic honey aashirvaad
17,hoodie,hoodie
179,GLUTEN FREE SPICES COMBO NESTLE,gluten free spice combo nestle
4,sugar free peanut butter,sugar free peanut butter
249,Unpolished Organic Honey Nestle,unpolished organic honey nestle
213,natural organic honey,natural organic honey
361,best smartphone!!,best smartphone


In [13]:
sample_raw = df['query_text'].iloc[10]
print('1. Raw:       ', repr(sample_raw))
print('2. Cleaned:   ', clean_text(sample_raw))
toks = word_tokenize(clean_text(sample_raw))
print('3. Tokenized: ', toks)
toks_ns = remove_stopwords(toks)
print('4. No stopwords:', toks_ns)
toks_lem = lemmatize_tokens(toks_ns)
print('5. Lemmatized:', toks_lem)

1. Raw:        'Smartphone Under 2000!!'
2. Cleaned:    smartphone under 2000
3. Tokenized:  ['smartphone', 'under', '2000']
4. No stopwords: ['smartphone', '2000']
5. Lemmatized: ['smartphone', '2000']


In [14]:
processed_texts = df['processed_text'].tolist()

bow_vectorizer = TextVectorization(output_mode='count', max_tokens=2000)
bow_vectorizer.adapt(processed_texts)                      # builds the vocabulary

vocab = bow_vectorizer.get_vocabulary()
print('Vocabulary size:', len(vocab))

Vocabulary size: 182


In [15]:
sample_queries = [processed_texts[10], processed_texts[50]]

for text in sample_queries:
    vec = bow_vectorizer([text])[0].numpy()
    nonzero = {vocab[i]: int(vec[i]) for i in np.nonzero(vec)[0]}
    print(f'Query: "{text}"  ->  BoW: {nonzero}')

Query: "smartphone 2000"  ->  BoW: {np.str_('2000'): 1, np.str_('smartphone'): 1}
Query: "power bank"  ->  BoW: {np.str_('power'): 1, np.str_('bank'): 1}


In [16]:
tfidf_vectorizer = TextVectorization(output_mode='tf_idf', max_tokens=2000)
tfidf_vectorizer.adapt(processed_texts)

tfidf_vocab = tfidf_vectorizer.get_vocabulary()

In [17]:
# Compare a common word vs. a rare word directly: how many queries each appears in,
# and the tf-idf weight that a single occurrence of that word gets.
common_word, rare_word = 'shoe', 'basmati'

for w in [common_word, rare_word]:
    if w in tfidf_vocab:
        doc_freq = sum(1 for t in processed_texts if w in t.split())
        tfidf_col = tfidf_vocab.index(w)
        single_word_weight = tfidf_vectorizer([w])[0, tfidf_col].numpy()
        print(f"'{w}': appears in {doc_freq} of {len(processed_texts)} queries "
              f"-> tf-idf weight per occurrence ~ {single_word_weight:.3f}")
    else:
        print(f"'{w}' not in vocabulary -- pick another word from `vocab` to compare")

'shoe': appears in 16 of 420 queries -> tf-idf weight per occurrence ~ 3.247
'basmati': appears in 5 of 420 queries -> tf-idf weight per occurrence ~ 4.263


In [18]:
sequence_length = int(np.percentile(df['processed_text'].str.split().apply(len), 95))
print('Chosen padding length (95th percentile of query length):', sequence_length)

int_vectorizer = TextVectorization(
    output_mode='int', max_tokens=2000, output_sequence_length=sequence_length)
int_vectorizer.adapt(processed_texts)

X_sequences = int_vectorizer(np.array(processed_texts)).numpy()
print('Padded sequence matrix shape:', X_sequences.shape)
print('\nExample:')
print('  text:    ', processed_texts[10])
print('  sequence:', X_sequences[10])

Chosen padding length (95th percentile of query length): 7
Padded sequence matrix shape: (420, 7)

Example:
  text:     smartphone 2000
  sequence: [85 74  0  0  0  0  0]


In [19]:
# Labels, ready for training a model on top of these sequences
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['category'])
class_names = label_encoder.classes_
vocab_size = int_vectorizer.vocabulary_size()
print('Vocabulary size for embeddings:', vocab_size)
print('Classes:', class_names)

Vocabulary size for embeddings: 183
Classes: ['Apparel' 'Electronics' 'Grocery']


In [20]:
embedding_dim = 16

embed_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length),
    GlobalAveragePooling1D(),          # average the word vectors in each query into one vector
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

embed_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
metrics=['accuracy'])
embed_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = embed_model.fit(X_sequences, y, epochs=30, validation_split=0.2, verbose=0)

print('Final training accuracy:  ', round(history.history['accuracy'][-1], 3))
print('Final validation accuracy:', round(history.history['val_accuracy'][-1], 3))

Final training accuracy:   1.0
Final validation accuracy: 1.0


In [22]:
embedding_weights = embed_model.layers[0].get_weights()[0]   # shape: (vocab_size, embedding_dim)
embed_vocab = int_vectorizer.get_vocabulary()

def word_vector(word):
    if word not in embed_vocab:
        return None
    return embedding_weights[embed_vocab.index(word)]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

pairs = [('shoe', 'sneaker'), ('cheap', 'budget'), ('organic', 'premium'),
         ('rice', 'sugar'), ('shoe', 'rice')]

for w1, w2 in pairs:
    v1, v2 = word_vector(w1), word_vector(w2)
    if v1 is not None and v2 is not None:
        print(f"cosine_similarity('{w1}', '{w2}') = {cosine_similarity(v1, v2):.3f}")
    else:
        print(f"'{w1}' or '{w2}' not in vocabulary -- try another pair from your dataset")

cosine_similarity('shoe', 'sneaker') = 0.983
cosine_similarity('cheap', 'budget') = 0.923
cosine_similarity('organic', 'premium') = 0.961
cosine_similarity('rice', 'sugar') = 0.971
cosine_similarity('shoe', 'rice') = -0.336


In [23]:
new_query = "  Best WIRELESS Earbuds under 1500!! "

cleaned = clean_text(new_query)
tokens = word_tokenize(cleaned)
tokens_ns = remove_stopwords(tokens)
tokens_lem = lemmatize_tokens(tokens_ns)
processed = ' '.join(tokens_lem)

int_seq = int_vectorizer([processed])
predicted_probs = embed_model.predict(int_seq, verbose=0)[0]
predicted_class = class_names[np.argmax(predicted_probs)]

print('1. Raw:        ', repr(new_query))
print('2. Cleaned:    ', cleaned)
print('3. Tokens:     ', tokens)
print('4. No stopword:', tokens_ns)
print('5. Lemmatized: ', tokens_lem)
print('9. Padded seq: ', int_seq.numpy()[0])
print('10. Predicted category:', predicted_class)

1. Raw:         '  Best WIRELESS Earbuds under 1500!! '
2. Cleaned:     best wireless earbuds under 1500
3. Tokens:      ['best', 'wireless', 'earbuds', 'under', '1500']
4. No stopword: ['best', 'wireless', 'earbuds', '1500']
5. Lemmatized:  ['best', 'wireless', 'earbuds', '1500']
9. Padded seq:  [102  25 158   1   0   0   0]
10. Predicted category: Electronics
